# First Attempt

In [1]:
import torch
import numpy as np

## Load/ Preprocess Data

In [2]:
import pandas as pd
import numpy as np
import wfdb
import ast

#provided function to load data with information in the agg_df (modified to return a numpy array)
def load_raw_data(df, sampling_rate, path):
    if sampling_rate == 100:
        data = [wfdb.rdsamp(path+f) for f in df.filename_lr]
    else:
        data = [wfdb.rdsamp(path+f) for f in df.filename_hr]
    data = np.array([signal for signal, meta in data]).astype(np.float32)
    return data

# path to dta and selected sampling rate
path = "physionet.org/files/ptb-xl/1.0.3/"
sampling_rate=100

# load and convert annotation data
Y = pd.read_csv(path+'ptbxl_database.csv', index_col='ecg_id')
Y.scp_codes = Y.scp_codes.apply(lambda x: ast.literal_eval(x)) #changes class distribution to dict

# Load scp_statements.csv for diagnostic aggregation
agg_df = pd.read_csv(path+'scp_statements.csv', index_col=0)
agg_df = agg_df[agg_df.diagnostic == 1] #only take diagnostic scps

classes = agg_df.index.tolist()

# encodes the labeled as a 44d vector of 1's and zeros (based off of classes list)
def encode_multilabel(class_list):
    return np.array([1 if cls in class_list else 0 for cls in classes])

def aggregate_diagnostic(y_dic):
    tmp = []
    for key in y_dic.keys():
        if key in agg_df.index:
            if y_dic[key] > 50:
                tmp.append(key)
    return list(set(tmp))

# Apply diagnostic superclass and encoding multilabel
Y['diagnostic_superclass'] = Y.scp_codes.apply(aggregate_diagnostic).apply(encode_multilabel)
Y['indv_diagnostic'] = Y.scp_codes.apply(aggregate_diagnostic).apply(encode_multilabel)
Y['indv_diagnostic_str'] = str(Y['indv_diagnostic'])

In [3]:
Y["diagnostic_superclass"]
Y

,patient_id,age,sex,height,weight,nurse,site,device,recording_date,report,...,burst_noise,electrodes_problems,extra_beats,pacemaker,strat_fold,filename_lr,filename_hr,diagnostic_superclass,indv_diagnostic,indv_diagnostic_str
ecg_id,,,,,,,,,,,,,,,,,,,,,
1,15709.0,56.0,1,NaN,63.0,2.0,0.0,CS-12 E,1984-11-09 09:17:34,sinusrhythmus periphere niederspannung,...,NaN,NaN,NaN,NaN,3,records100/00000/00001_lr,records500/00000/00001_hr,"[0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","ecg_id\n1 [0, 0, 0, 0, 1, 0, 0, 0, 0, 0..."
2,13243.0,19.0,0,NaN,70.0,2.0,0.0,CS-12 E,1984-11-14 12:55:37,sinusbradykardie sonst normales ekg,...,NaN,NaN,NaN,NaN,2,records100/00000/00002_lr,records500/00000/00002_hr,"[0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","ecg_id\n1 [0, 0, 0, 0, 1, 0, 0, 0, 0, 0..."
3,20372.0,37.0,1,NaN,69.0,2.0,0.0,CS-12 E,1984-11-15 12:49:10,sinusrhythmus normales ekg,...,NaN,NaN,NaN,NaN,5,records100/00000/00003_lr,records500/00000/00003_hr,"[0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","ecg_id\n1 [0, 0, 0, 0, 1, 0, 0, 0, 0, 0..."
4,17014.0,24.0,0,NaN,82.0,2.0,0.0,CS-12 E,1984-11-15 13:44:57,sinusrhythmus normales ekg,...,NaN,NaN,NaN,NaN,3,records100/00000/00004_lr,records500/00000/00004_hr,"[0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","ecg_id\n1 [0, 0, 0, 0, 1, 0, 0, 0, 0, 0..."
5,17448.0,19.0,1,NaN,70.0,2.0,0.0,CS-12 E,1984-11-17 10:43:15,sinusrhythmus normales ekg,...,NaN,NaN,NaN,NaN,4,records100/00000/00005_lr,records500/00000/00005_hr,"[0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","ecg_id\n1 [0, 0, 0, 0, 1, 0, 0, 0, 0, 0..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21833,17180.0,67.0,1,NaN,NaN,1.0,2.0,AT-60 3,2001-05-31 09:14:35,ventrikulÄre extrasystole(n) sinustachykardie ...,...,NaN,NaN,1ES,NaN,7,records100/21000/21833_lr,records500/21000/21833_hr,"[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","ecg_id\n1 [0, 0, 0, 0, 1, 0, 0, 0, 0, 0..."
21834,20703.0,300.0,0,NaN,NaN,1.0,2.0,AT-60 3,2001-06-05 11:33:39,sinusrhythmus lagetyp normal qrs(t) abnorm ...,...,NaN,NaN,NaN,NaN,4,records100/21000/21834_lr,records500/21000/21834_hr,"[0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","ecg_id\n1 [0, 0, 0, 0, 1, 0, 0, 0, 0, 0..."
21835,19311.0,59.0,1,NaN,NaN,1.0,2.0,AT-60 3,2001-06-08 10:30:27,sinusrhythmus lagetyp normal t abnorm in anter...,...,NaN,NaN,NaN,NaN,2,records100/21000/21835_lr,records500/21000/21835_hr,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","ecg_id\n1 [0, 0, 0, 0, 1, 0, 0, 0, 0, 0..."


In [4]:
(Y["indv_diagnostic"].apply(tuple).value_counts())
Y_matrix = np.stack(Y.indv_diagnostic.values).astype(np.float32)
Y_cond = pd.DataFrame(Y_matrix)
Y_cond = Y_cond.loc[:, Y_cond.any()]

In [5]:
Y.scp_codes

ecg_id
1                 {'NORM': 100.0, 'LVOLT': 0.0, 'SR': 0.0}
2                             {'NORM': 80.0, 'SBRAD': 0.0}
3                               {'NORM': 100.0, 'SR': 0.0}
4                               {'NORM': 100.0, 'SR': 0.0}
5                               {'NORM': 100.0, 'SR': 0.0}
                               ...                        
21833    {'NDT': 100.0, 'PVC': 100.0, 'VCLVH': 0.0, 'ST...
21834             {'NORM': 100.0, 'ABQRS': 0.0, 'SR': 0.0}
21835                           {'ISCAS': 50.0, 'SR': 0.0}
21836                           {'NORM': 100.0, 'SR': 0.0}
21837                           {'NORM': 100.0, 'SR': 0.0}
Name: scp_codes, Length: 21799, dtype: object

In [6]:
agg_df.index

Index(['NDT', 'NST_', 'DIG', 'LNGQT', 'NORM', 'IMI', 'ASMI', 'LVH', 'LAFB',
       'ISC_', 'IRBBB', '1AVB', 'IVCD', 'ISCAL', 'CRBBB', 'CLBBB', 'ILMI',
       'LAO/LAE', 'AMI', 'ALMI', 'ISCIN', 'INJAS', 'LMI', 'ISCIL', 'LPFB',
       'ISCAS', 'INJAL', 'ISCLA', 'RVH', 'ANEUR', 'RAO/RAE', 'EL', 'WPW',
       'ILBBB', 'IPLMI', 'ISCAN', 'IPMI', 'SEHYP', 'INJIN', 'INJLA', 'PMI',
       '3AVB', 'INJIL', '2AVB'],
      dtype='str')

In [7]:
# Further example code 
"""
Y_s = Y.head(10).copy()

# Load raw signal data
X = load_raw_data(Y_s, sampling_rate, path)

# Split data into train and test
test_fold = 10
# Train
X_train = X[np.where(Y_s.strat_fold != test_fold)]
y_train = Y_s[(Y_s.strat_fold != test_fold)].diagnostic_superclass
# Test
X_test = X[np.where(Y_s.strat_fold == test_fold)]
y_test = Y_s[Y_s.strat_fold == test_fold].diagnostic_superclass"""

'\nY_s = Y.head(10).copy()\n\n# Load raw signal data\nX = load_raw_data(Y_s, sampling_rate, path)\n\n# Split data into train and test\ntest_fold = 10\n# Train\nX_train = X[np.where(Y_s.strat_fold != test_fold)]\ny_train = Y_s[(Y_s.strat_fold != test_fold)].diagnostic_superclass\n# Test\nX_test = X[np.where(Y_s.strat_fold == test_fold)]\ny_test = Y_s[Y_s.strat_fold == test_fold].diagnostic_superclass'

In [8]:
#Y_s.diagnostic_superclass.to_numpy() #example line run of multilabel distribution
#Y['scp_codes']

In [9]:
Y['scp_codes']

ecg_id
1                 {'NORM': 100.0, 'LVOLT': 0.0, 'SR': 0.0}
2                             {'NORM': 80.0, 'SBRAD': 0.0}
3                               {'NORM': 100.0, 'SR': 0.0}
4                               {'NORM': 100.0, 'SR': 0.0}
5                               {'NORM': 100.0, 'SR': 0.0}
                               ...                        
21833    {'NDT': 100.0, 'PVC': 100.0, 'VCLVH': 0.0, 'ST...
21834             {'NORM': 100.0, 'ABQRS': 0.0, 'SR': 0.0}
21835                           {'ISCAS': 50.0, 'SR': 0.0}
21836                           {'NORM': 100.0, 'SR': 0.0}
21837                           {'NORM': 100.0, 'SR': 0.0}
Name: scp_codes, Length: 21799, dtype: object

## Data Prep

In [10]:
from torch.utils.data import WeightedRandomSampler

condition_freq = Y_cond.sum(axis=0)
condition_inv_freq = 1.0 / (condition_freq + 1e-6)

row_weights = (Y_cond * condition_inv_freq).sum(axis=1)

sampler = WeightedRandomSampler(
    weights=row_weights,
    num_samples=len(row_weights),
    replacement = True
)

In [11]:
pos_weight = (len(Y) - condition_freq) / condition_freq
loss = torch.nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pos_weight))

In [12]:
Y_cond['filename_lr'] = Y['filename_lr'].to_numpy()
Y_cond = Y_cond[~(Y_cond.drop(columns=['filename_lr'])==0).all(axis=1)]
Y_cond

,0,1,2,3,4,5,6,7,8,9,...,35,36,37,38,39,40,41,42,43,filename_lr
0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,records100/00000/00001_lr
1,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,records100/00000/00002_lr
2,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,records100/00000/00003_lr
3,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,records100/00000/00004_lr
4,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,records100/00000/00005_lr
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21793,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,records100/21000/21832_lr
21794,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,records100/21000/21833_lr
21795,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,records100/21000/21834_lr
21797,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,records100/21000/21836_lr


In [13]:
from sklearn.model_selection import train_test_split

Y_transformed = np.stack(Y.diagnostic_superclass.values).astype(np.float32)

# Load raw signal data
X = load_raw_data(Y, sampling_rate, path)

X = np.transpose(X, (0, 2, 1))

X_train, X_test, y_train, y_test = train_test_split(X, Y_transformed, test_size=0.1, random_state=56)

In [14]:
X_train_torch = torch.tensor(X_train) # change dim ordering for PyTorch
y_train_torch = torch.tensor(y_train)
X_test_torch = torch.tensor(X_test) # change dim ordering for PyTorch
y_test_torch = torch.tensor(y_test)

In [15]:
from torch.utils.data import WeightedRandomSampler

condition_freq = y_train_torch.sum(axis=0)
condition_inv_freq = 1.0 / np.log((condition_freq + 1e-6))

row_weights = (y_train_torch * condition_inv_freq).sum(axis=1)
row_weights = row_weights / row_weights.sum()
sampler = WeightedRandomSampler(
    weights=row_weights,
    num_samples=len(row_weights),
    replacement = True
)

/tmp/ipykernel_1849/3358833782.py:4: DeprecationWarning: __array_wrap__ must accept context and return_scalar arguments (positionally) in the future. (Deprecated NumPy 2.0)
  condition_inv_freq = 1.0 / np.log((condition_freq + 1e-6))


In [16]:
train_dataset = torch.utils.data.TensorDataset(X_train_torch, y_train_torch)
train_dataloader = torch.utils.data.DataLoader(dataset=train_dataset, batch_size=32, sampler=sampler)

test_dataset = torch.utils.data.TensorDataset(X_test_torch, y_test_torch)
test_dataloader = torch.utils.data.DataLoader(dataset=test_dataset, batch_size=32)


In [17]:
# train_dataset = torch.utils.data.TensorDataset(X_train_torch, y_train_torch)
# train_dataloader = torch.utils.data.DataLoader(dataset=train_dataset, batch_size=32, shuffle=True)

# test_dataset = torch.utils.data.TensorDataset(X_test_torch, y_test_torch)
# test_dataloader = torch.utils.data.DataLoader(dataset=test_dataset, batch_size=32)


## CNN

In [18]:
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F


class CNet(nn.Module):
  def __init__(self):
    super(CNet,self).__init__()
    #starting with 96

    self.conv11 = nn.Conv1d(12, 64, 9, padding=4)
    self.conv12 = nn.Conv1d(64, 64, 9, padding=4)
    self.bn1 = nn.BatchNorm1d(64)

    self.pool = nn.MaxPool1d(kernel_size=2, stride = 2)

    self.conv21 = nn.Conv1d(64, 128, 9, padding=4)
    self.conv22 = nn.Conv1d(128, 128, 9, padding=4)
    self.bn2 = nn.BatchNorm1d(128)

    self.conv31 = nn.Conv1d(128, 256, 9, padding=4)
    self.conv32 = nn.Conv1d(256, 256, 9, padding=4)
    self.conv33 = nn.Conv1d(256, 256, 9, padding=4)
    self.bn3 = nn.BatchNorm1d(256)

    self.conv41 = nn.Conv1d(256, 512, 9, padding=4)
    self.conv42 = nn.Conv1d(512, 512, 9, padding=4)
    self.conv43 = nn.Conv1d(512, 512, 9, padding=4)
    self.bn4 = nn.BatchNorm1d(512)

    self.fc1 = nn.Linear(512 * 125, 1000)
    self.fc2 = nn.Linear(1000, 500)
    self.fc3 = nn.Linear(500, 44)

  def forward(self, x):
    #x = self.pool(F.relu(self.conv12(F.relu(self.conv11(x)))))
    #x = self.pool(F.relu(self.conv22(F.relu(self.conv21(x)))))
    #x = self.pool(F.relu(self.conv33(F.relu(self.conv32(F.relu(self.conv31(x)))))))
    #x = F.relu(self.conv43(F.relu(self.conv42(F.relu(self.conv41(x))))))

    x = self.pool(F.relu(self.bn1(self.conv12(F.relu(self.conv11(x))))))
    x = self.pool(F.relu(self.bn2(self.conv22(F.relu(self.conv21(x))))))
    x = self.pool(F.relu(self.bn3(self.conv33(F.relu(self.conv32(F.relu(self.conv31(x))))))))
    x = F.relu(self.bn4(self.conv43(F.relu(self.conv42(F.relu(self.conv41(x)))))))
    x = torch.flatten(x,1)
    x = F.relu(self.fc1(x))
    x = F.relu(self.fc2(x))
    x = self.fc3(x)
    return x

In [19]:
# import torch.nn as nn
# import torch.optim as optim
# import torch.nn.functional as F


# class CNet(nn.Module):
#   def __init__(self):
#     super(CNet,self).__init__()
#     #starting with 96

#     self.conv11 = nn.Conv1d(12, 64, 21, padding=10)
#     self.conv12 = nn.Conv1d(64, 64, 21, padding=10)
#     self.bn1 = nn.BatchNorm1d(64)

#     self.pool = nn.MaxPool1d(kernel_size=2, stride = 2)

#     self.conv21 = nn.Conv1d(64, 128, 21, padding=10)
#     self.conv22 = nn.Conv1d(128, 128, 21, padding=10)
#     self.bn2 = nn.BatchNorm1d(128)

#     self.conv31 = nn.Conv1d(128, 256, 21, padding=10)
#     self.conv32 = nn.Conv1d(256, 256, 21, padding=10)
#     self.conv33 = nn.Conv1d(256, 256, 21, padding=10)
#     self.bn3 = nn.BatchNorm1d(256)

#     self.fc1 = nn.Linear(256 * 250, 1000)
#     self.fc2 = nn.Linear(1000, 500)
#     self.fc3 = nn.Linear(500, 44)

#   def forward(self, x):
#     #x = self.pool(F.relu(self.conv12(F.relu(self.conv11(x)))))
#     #x = self.pool(F.relu(self.conv22(F.relu(self.conv21(x)))))
#     #x = self.pool(F.relu(self.conv33(F.relu(self.conv32(F.relu(self.conv31(x)))))))
#     #x = F.relu(self.conv43(F.relu(self.conv42(F.relu(self.conv41(x))))))

#     x = self.pool(F.relu(self.bn1(self.conv12(F.relu(self.conv11(x))))))
#     x = self.pool(F.relu(self.bn2(self.conv22(F.relu(self.conv21(x))))))
#     x = F.relu(self.bn3(self.conv33(F.relu(self.conv32(F.relu(self.conv31(x)))))))
#     x = torch.flatten(x,1)
#     x = F.relu(self.fc1(x))
#     x = F.relu(self.fc2(x))
#     x = self.fc3(x)
#     return x

In [20]:
def m_train_model(model, train_dataloader, test_dataloader, epochs, weight_decay):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)

    model.train()

    #loss_fn = torch.nn.CrossEntropyLoss()
    #loss_fn = nn.BCEWithLogitsLoss(pos_weight=torch.Tensor([5]).to(device))

    # MASKED LOSS 
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=torch.Tensor([5]).to(device), reduction="none").to(device)

    lr = 1e-3
    #opt = torch.optim.SGD(model.parameters(), lr=lr, weight_decay=weight_decay)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)

    best_test = 0
    best_test_train = 0
    best_epoch = 0

    for epoch in range(epochs):
        running_loss = 0
        for batch_index, (X, y) in enumerate(train_dataloader):
            #print(f"Epoch: {epoch} Batch {batch_index}")
            X,y = X.to(device), y.to(device)

            opt.zero_grad() #zero out the gradients

            z = model(X)
            loss = loss_fn(z,y) #compute loss
            weights = torch.ones(y.shape[1], device=device, dtype=loss.dtype)
            weights[4] = 0.1
            weighted_loss = (loss * weights.unsqueeze(0))
            #running_loss += loss.item()

            final_loss = weighted_loss.sum() / (weights.sum() * X.size(0))
            running_loss += final_loss.item()
            final_loss.backward()
            #loss.backward() #compute gradients

            opt.step() #apply gradients
        scheduler.step()



        running_loss = running_loss / len(train_dataloader)
        print(f"Epoch {epoch} train loss: {running_loss}")

def compute_pos_weight(train_dataloader, num_labels, device):
    pos = torch.zeros(num_labels, dtype=torch.float32)
    total = 0

    for _, y in train_dataloader:
        pos += y.float().sum(dim=0)
        total += y.shape[0]

    neg = total - pos
    pos_weight = neg / (pos + 1e-6)

    #pos_weight = pos_weight.clamp(1.0, 20.0)

    return pos_weight.to(device)

def train_model(model, train_dataloader, test_dataloader, epochs, weight_decay):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)

    model.train()

    #loss_fn = torch.nn.CrossEntropyLoss()
    #loss_fn = nn.BCEWithLogitsLoss(pos_weight=compute_pos_weight(train_dataloader, 44, device))
    # f = torch.tensor(y_train).sum(dim=0)
    # N = y_train.shape[0]
    # pos_weight = (N - f) / (f + 1e-6)
    # pos_weight = (torch.log(pos_weight)).to(device)
    # loss_fn = torch.nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pos_weight))
    loss_fn = nn.BCEWithLogitsLoss()
    lr = 1e-3
    #opt = torch.optim.SGD(model.parameters(), lr=lr, weight_decay=weight_decay)
    #loss_fn = torch.nn.BCEWithLogitsLoss()
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)

    best_test = 0
    best_test_train = 0
    best_epoch = 0

    for epoch in range(epochs):
        running_loss = 0
        for batch_index, (X, y) in enumerate(train_dataloader):
            #print(f"Epoch: {epoch} Batch {batch_index}")
            X,y = X.to(device), y.to(device)

            opt.zero_grad() #zero out the gradients

            z = model(X)
            loss = loss_fn(z,y) #compute loss
            running_loss += loss.item()

            loss.backward() #compute gradients

            opt.step() #apply gradients
        scheduler.step()



        running_loss = running_loss / len(train_dataloader)
        print(f"Epoch {epoch} train loss: {running_loss}")


In [21]:
X.shape

(21799, 12, 1000)

In [22]:
def my_jaccard_score(y_pred, y_true):
    intersection = ((y_pred == 1) & (y_true == 1)).sum()
    union = ((y_pred == 1) | (y_true == 1)).sum()

    if union == 0:
        return 0.0
    
    return intersection / union

def precision(y_pred, y_true):
    f_p = ((y_pred == 1) & (y_true == 1)).sum()
    t_p = ((y_pred == 1) & (y_true == 0)).sum()
    if t_p + f_p == 0:
        return 0.0
    return t_p / (t_p + f_p)

def recall(y_pred, y_true):
    t_p = ((y_pred == 1) & (y_true == 1)).sum()
    f_n = ((y_pred == 0) & (y_true == 1)).sum()

    if t_p + f_n == 0:
        return 0.0
    return t_p / (t_p + f_n)

def f1(y_pred, y_true):
    m_p = precision(y_pred, y_true)
    m_r = recall(y_pred, y_true)
    if m_p + m_r == 0:
        return 0.0
    return 2 * (m_p*m_r) / (m_p+m_r)

def macro_precision(y_pred, y_true):
    y_pred = y_pred.int()
    y_true = y_true.int()
    t_p = ((y_pred == 1) & (y_true == 1)).sum(dim=0)
    f_p = ((y_pred == 1) & (y_true == 0)).sum(dim=0)
    return t_p.float() / (t_p + f_p).float().clamp(min=1e-8)

def macro_recall(y_pred, y_true):
    y_pred = y_pred.int()
    y_true = y_true.int()
    t_p = ((y_pred == 1) & (y_true == 1)).sum(dim=0)
    f_n = ((y_pred == 0) & (y_true == 1)).sum(dim=0)
    return t_p.float() / (t_p + f_n).float().clamp(min=1e-8)

def macro_f1(y_pred, y_true):
    m_p = macro_precision(y_pred, y_true)
    m_r = recall(y_pred, y_true)
    return 2 * m_p * m_r / (m_p + m_r).clamp(min=1e-8)


In [23]:
device = 'cuda'

In [ ]:
net = CNet().to(device)
train_model(net, train_dataloader, test_dataloader, 30, 1e-3)

Epoch 0 train loss: 0.14116931165468422
Epoch 1 train loss: 0.12037802928093978
Epoch 2 train loss: 0.11347186135838008
Epoch 3 train loss: 0.10832197651121438
Epoch 4 train loss: 0.10514165749864392
Epoch 5 train loss: 0.10204300932367773
Epoch 6 train loss: 0.10060839307060654
Epoch 7 train loss: 0.09970768984006748
Epoch 8 train loss: 0.09900141322942821
Epoch 9 train loss: 0.09867604707856131
Epoch 10 train loss: 0.09641230346616782
Epoch 11 train loss: 0.09619527986631138
Epoch 12 train loss: 0.09508590233369642
Epoch 13 train loss: 0.09435623434011245
Epoch 14 train loss: 0.09456990201766406


In [ ]:
from sklearn.metrics import jaccard_score

@torch.no_grad()
def dig_accurate(y_preds, y_ground):
    y_preds_cpu = y_preds.detach().to("cpu", dtype=torch.int32)
    y_ground_cpu = y_ground.detach().to("cpu", dtype=torch.int32)
    element_wise_eq = y_preds_cpu == y_ground_cpu
    y_preds = y_preds_cpu.numpy()
    y_ground = y_ground_cpu.numpy()

    row_acc = torch.all(element_wise_eq, dim=1).float().mean().item()
    elem_acc = element_wise_eq.float().mean().item()
    print(f"Jaccard_Score: {jaccard_score(y_ground, y_preds, average="samples")}")
    print(f"Row-wise Accuracy (perfect match): {row_acc}")
    print(f"Element Wise Accuracy: {elem_acc}")

In [ ]:
net.eval()
out = (torch.sigmoid(net(X_test_torch.to("cuda"))) > 0.5).detach().to("cpu", dtype=torch.int32)
input = y_test_torch.to("cpu", dtype=torch.int32)
print(dig_accurate(out, input))

Jaccard_Score: 0.44308103975535174
Row-wise Accuracy (perfect match): 0.4802752435207367
Element Wise Accuracy: 0.9827772974967957
None


/home/tpham/torchenv312/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Jaccard is ill-defined and being set to 0.0 in samples with no true or predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [ ]:
net.eval()

with torch.no_grad():
    logits = net(X_test_torch.to("cuda"))
    probs = torch.sigmoid(logits)

input = y_test_torch.to("cuda").int()

best_thresh = None
best_score = -1

for t in torch.arange(0.0, 1.01, 0.05, device="cuda"):
    preds = (probs > t).int()

    print(f"\nThreshold: {t.item():.2f}")
    dig_accurate(preds, input)


Threshold: 0.00
Jaccard_Score: 0.025594245204336948
Row-wise Accuracy (perfect match): 0.0
Element Wise Accuracy: 0.02559424564242363

Threshold: 0.05
Jaccard_Score: 0.3787540148434644
Row-wise Accuracy (perfect match): 0.23440366983413696
Element Wise Accuracy: 0.9027418494224548

Threshold: 0.10
Jaccard_Score: 0.45870674305307335
Row-wise Accuracy (perfect match): 0.30596330761909485
Element Wise Accuracy: 0.9484882950782776

Threshold: 0.15
Jaccard_Score: 0.49359582059123347
Row-wise Accuracy (perfect match): 0.3559632897377014
Element Wise Accuracy: 0.963042140007019

Threshold: 0.20
Jaccard_Score: 0.5144440803844473
Row-wise Accuracy (perfect match): 0.4013761579990387
Element Wise Accuracy: 0.9707673192024231

Threshold: 0.25
Jaccard_Score: 0.5199999999999999
Row-wise Accuracy (perfect match): 0.43715596199035645
Element Wise Accuracy: 0.9755525588989258

Threshold: 0.30
Jaccard_Score: 0.5273318042813456
Row-wise Accuracy (perfect match): 0.4724770784378052
Element Wise Accuracy

/home/tpham/torchenv312/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Jaccard is ill-defined and being set to 0.0 in samples with no true or predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/tpham/torchenv312/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Jaccard is ill-defined and being set to 0.0 in samples with no true or predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/tpham/torchenv312/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Jaccard is ill-defined and being set to 0.0 in samples with no true or predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is

In [ ]:
net.eval()
count = 0
j_score = 0.0
f1_score = 0.0
p_score = 0.0
r_score = 0.0

a_m_f1_score = []
a_m_p_score = []
a_m_r_score = []


for i, (x_test, y_test) in enumerate(test_dataloader):
    # if i > 2: 
    #     break
    val = (torch.sigmoid(net(x_test.to(device))) > 0.3).int()
    count += 1
    y_test = y_test.to(device).int()
    j_score += my_jaccard_score(val, y_test)
    f1_score += f1(val, y_test)
    p_score += precision(val, y_test)
    r_score += recall(val, y_test)
    a_m_f1_score.append(macro_f1(val, y_test))
    a_m_p_score.append(macro_precision(val, y_test))
    a_m_r_score.append(macro_recall(val, y_test))

a_m_f1_score = torch.stack(a_m_f1_score).cpu()
a_m_p_score = torch.stack(a_m_p_score).cpu()
a_m_r_score = torch.stack(a_m_r_score).cpu()
print(a_m_f1_score)
print(a_m_p_score)
print(a_m_r_score)
print(j_score/count, f1_score/count, p_score/count, r_score/count)
print(a_m_f1_score.mean(), a_m_p_score.mean(), a_m_r_score.mean())

tensor([[0.3056, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
        [0.3621, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
        [0.3193, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
        ...,
        [0.4752, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
        [0.7500, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000]])
tensor([[0.2000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
        [0.2500, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
        [0.2500, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
        ...,
        [0.3750, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
        [1.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000]])
tensor([[0.3333, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
        [1.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
        [0.5000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
        ...,

In [ ]:
net.eval()
count = 0
j_score = 0.0
f1_score = 0.0
p_score = 0.0
r_score = 0.0

a_m_f1_score = []
a_m_p_score = []
a_m_r_score = []


for i, (x_test, y_test) in enumerate(train_dataloader):
    # if i > 2: 
    #     break
    val = (torch.sigmoid(net(x_test.to(device))) > 0.5).int()
    count += 1
    y_test = y_test.to(device).int()
    j_score += jaccard_score(val, y_test)
    f1_score += f1(val, y_test)
    p_score += precision(val, y_test)
    r_score += recall(val, y_test)
    a_m_f1_score.append(macro_f1(val, y_test))
    a_m_p_score.append(macro_precision(val, y_test))
    a_m_r_score.append(macro_recall(val, y_test))

a_m_f1_score = torch.stack(a_m_f1_score).cpu()
a_m_p_score = torch.stack(a_m_p_score).cpu()
a_m_r_score = torch.stack(a_m_r_score).cpu()
print(a_m_f1_score)
print(a_m_p_score)
print(a_m_r_score)
print(j_score/count, f1_score/count, p_score/count, r_score/count)

TypeError: can't convert cuda:0 device type tensor to numpy. Use Tensor.cpu() to copy the tensor to host memory first.

In [ ]:
net.eval()
all_probs = []
all_targets = []

with torch.no_grad():
    for x_batch, y_batch in test_dataloader:
        x_batch = x_batch.to(device)
        logits = net(x_batch)
        probs = torch.sigmoid(logits).cpu().numpy()

        all_probs.append(probs)
        all_targets.append(y_batch.cpu().numpy())

y_val_prob = np.vstack(all_probs)     # shape (n_samples, n_labels)
y_val_true = np.vstack(all_targets) 

In [ ]:
import numpy as np
from sklearn.metrics import f1_score

def find_best_thresholds(y_true, y_prob, grid=None):
    """
    y_true: (n_samples, n_labels) in {0,1}
    y_prob: (n_samples, n_labels) in [0,1]
    """
    n_labels = y_true.shape[1]

    if grid is None:
        grid = np.concatenate([
            np.arange(0.01, 0.10, 0.01),
            np.arange(0.10, 0.50, 0.02),
            np.arange(0.50, 0.95, 0.05),
        ])

    best_thresholds = np.full(n_labels, 0.5, dtype=np.float32)
    best_f1s = np.zeros(n_labels, dtype=np.float32)

    for j in range(n_labels):
        y_j = y_true[:, j]
        p_j = y_prob[:, j]

        # if no positives in validation, threshold tuning is not meaningful
        if y_j.sum() == 0:
            best_thresholds[j] = 0.5
            best_f1s[j] = 0.0
            continue

        best_f1 = -1.0
        best_t = 0.5

        for t in grid:
            pred_j = (p_j >= t).astype(int)
            f1 = f1_score(y_j, pred_j, zero_division=0)

            if f1 > best_f1:
                best_f1 = f1
                best_t = t

        best_thresholds[j] = best_t
        best_f1s[j] = best_f1

    return best_thresholds, best_f1s

In [ ]:
thresholds, val_f1s = find_best_thresholds(y_val_true, y_val_prob)

In [ ]:
net.eval()
count = 0
j_score = 0.0
f1_score = 0.0
p_score = 0.0
r_score = 0.0

a_m_f1_score = []
a_m_p_score = []
a_m_r_score = []

for x_test, y_test in test_dataloader:
    probs = torch.sigmoid(net(x_test.to(device)))
    thresholds_t = torch.tensor(thresholds, device=device)
    val = (probs > thresholds_t).int()
    count += 1
    y_test = y_test.to(device).int()
    j_score += jaccard_score(val, y_test)
    f1_score += f1(val, y_test)
    p_score += precision(val, y_test)
    r_score += recall(val, y_test)
    a_m_f1_score.append(macro_f1(val, y_test))
    a_m_p_score.append(macro_precision(val, y_test))
    a_m_r_score.append(macro_recall(val, y_test))

a_m_f1_score = torch.stack(a_m_f1_score)
a_m_p_score = torch.stack(a_m_p_score)
a_m_r_score = torch.stack(a_m_r_score)
print(a_m_f1_score)
print(a_m_p_score)
print(a_m_r_score)
print(j_score/count, f1_score/count, p_score/count, r_score/count)

In [ ]:
# def dig_accurate(y_preds, y_ground):
#     y_preds = y_preds.cpu()
#     y_ground = y_ground.cpu()
#     # y_preds[:, 4] = 0
#     element_wise_eq = y_preds == y_ground

#     # for r in y_preds[torch.all(element_wise_eq, dim=1).int() == 1]:
#     #     if (r.sum() > 0):
#     #         print(r)
#     print(y_preds[torch.all(element_wise_eq, dim=1).int() == 1][0])

#     print(f"Row-wise Accuracy (perfect match): {sum(torch.all(element_wise_eq, dim=1).int())/element_wise_eq.shape[0]}")
#     print(f"Element Wise Accuracy: {sum(element_wise_eq.int().flatten(0))/torch.numel(element_wise_eq)}")

In [ ]:
# print(dig_accurate((torch.sigmoid(net(X_test_torch.to(device))) > 0.9).int(), y_test_torch.int()))